# Notebook 02 — Text Preprocessing

In [1]:
# Switch to GPU if present
import torch

print(torch.backends.mps.is_available())
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

x = torch.randn(5, 5).to(device)
print(x.device)

True
mps:0


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
tqdm.pandas()

from src.config import RESULTS_DIR
from src.data_loader import load_reviews, create_splits
from src.preprocessing import clean_text, tokenize_and_remove_stopwords, preprocess, build_tfidf

## 1 · Load Stratified Sample

In [3]:
BASELINE_SAMPLE_SIZE = 200_000
print(f'Loading {BASELINE_SAMPLE_SIZE:,} stratified reviews...')
df = load_reviews(sample_size=BASELINE_SAMPLE_SIZE)

print(f'\nDataset shape: {df.shape}')
print(f'Label distribution:\n{df["label"].value_counts()}')

Loading 200,000 stratified reviews...

Dataset shape: (200000, 8)
Label distribution:
label
0    147405
1     52595
Name: count, dtype: int64


## 2 · Text Cleaning — Before & After

In [4]:
# Show before/after for a few examples
examples = df.sample(5, random_state=42)
for i, row in examples.iterrows():
    raw = row['review_text'][:300]
    cleaned = clean_text(row['review_text'])[:300]
    print(f'--- Example (spoiler={row["is_spoiler"]}) ---')
    print(f'RAW   : {raw}...')
    print(f'CLEAN : {cleaned}...')
    print()

--- Example (spoiler=False) ---
RAW   : First, let me say that I never watched the Avatar TV series. Second, I had virtually no trouble figuring out what was going on. Third, while it's true that there are some cheesy moments and a couple of characters are woefully underdeveloped, overall it was not a bad movie experience. The acting was ...
CLEAN : first let me say that i never watched the avatar tv series second i had virtually no trouble figuring out what was going on third while it s true that there are some cheesy moments and a couple of characters are woefully underdeveloped overall it was not a bad movie experience the acting was not the...

--- Example (spoiler=True) ---
RAW   : Many of us, not least those who populate the online world with movie reviews, observations and criticisms, are aware of Chris Nolan's raw talent. Memento broke onto the screen a decade ago and provided us with a spectacle in story telling so rare that it dumbfounded us. Since then, Nolan has gained ...


## 3 · Full Preprocessing Pipeline

In [5]:
print('Running preprocessing pipeline')
df['processed_text'] = df['review_text'].progress_apply(preprocess)

# Show a sample
print('\n--- Preprocessed Sample ---')
for i, row in df.head(3).iterrows():
    print(f'[{"SPOILER" if row["is_spoiler"] else "NON-SPOILER"}] {row["processed_text"][:200]}...')
    print()

Running preprocessing pipeline


100%|██████████| 200000/200000 [01:08<00:00, 2900.52it/s]


--- Preprocessed Sample ---
[NON-SPOILER] open heart floats hope life full twist turns control lived past age never unforeseen circumstances thrust upon actions someone else rude awakening one day coming allow heart open good come way survive...

[SPOILER] seen enough original dukes hazzard know mean original series tv classic living forever reruns american icon first heard yet another remake saw cast help shudder specifically came bo luke duke horrifie...

[NON-SPOILER] figured movie would pretty weak even got hey correct one part stuck mind jack supposed year old white owner hip hop label reduced rap commented diane keaton character white found conversation offensiv...



In [6]:
print(df.columns.tolist())

['review_date', 'movie_id', 'user_id', 'is_spoiler', 'review_text', 'rating', 'review_summary', 'label', 'processed_text']


## 4 · Train / Validation / Test Splits

In [7]:
train_df, val_df, test_df = create_splits(df)

print(f'Train : {len(train_df):,}  (spoiler: {train_df["label"].mean():.1%})')
print(f'Val   : {len(val_df):,}  (spoiler: {val_df["label"].mean():.1%})')
print(f'Test  : {len(test_df):,}  (spoiler: {test_df["label"].mean():.1%})')

Train : 140,000  (spoiler: 26.3%)
Val   : 20,000  (spoiler: 26.3%)
Test  : 40,000  (spoiler: 26.3%)


## 5 · TF-IDF Vectorization

In [8]:
tfidf_data = build_tfidf(
    train_df['processed_text'],
    val_df['processed_text'],
    test_df['processed_text'],
)

print(f'Vocab size      : {len(tfidf_data["vectorizer"].vocabulary_):,}')
print(f'X_train shape   : {tfidf_data["X_train"].shape}')
print(f'X_val shape     : {tfidf_data["X_val"].shape}')
print(f'X_test shape    : {tfidf_data["X_test"].shape}')

Vocab size      : 50,000
X_train shape   : (140000, 50000)
X_val shape     : (20000, 50000)
X_test shape    : (40000, 50000)


In [9]:
# Top TF-IDF features
feature_names = tfidf_data['vectorizer'].get_feature_names_out()
mean_tfidf = tfidf_data['X_train'].mean(axis=0).A1
top_idx = mean_tfidf.argsort()[::-1][:20]

print('Top 20 TF-IDF features (by mean score):')
for i, idx in enumerate(top_idx, 1):
    print(f'  {i:2d}. {feature_names[idx]:25s} ({mean_tfidf[idx]:.4f})')

Top 20 TF-IDF features (by mean score):
   1. movie                     (0.0300)
   2. film                      (0.0239)
   3. one                       (0.0190)
   4. like                      (0.0174)
   5. good                      (0.0165)
   6. story                     (0.0149)
   7. great                     (0.0145)
   8. see                       (0.0143)
   9. really                    (0.0142)
  10. time                      (0.0139)
  11. would                     (0.0131)
  12. well                      (0.0131)
  13. even                      (0.0124)
  14. much                      (0.0123)
  15. movies                    (0.0122)
  16. first                     (0.0116)
  17. best                      (0.0115)
  18. people                    (0.0114)
  19. also                      (0.0112)
  20. characters                (0.0110)


## 6 · Save Preprocessed Data

In [10]:
import pickle

preprocessed = {
    'train_df': train_df,
    'val_df': val_df,
    'test_df': test_df,
    'tfidf_data': tfidf_data,
}

save_path = os.path.join(RESULTS_DIR, 'preprocessed_data.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(preprocessed, f)

print(f'Preprocessed data saved → {save_path}')

Preprocessed data saved → /Users/hardikk/Desktop/spoiler-detection-nlp/results/preprocessed_data.pkl
